In [1]:
"""
Logistic regression + gradient boosting on its errors (boosting.py).

Step 1 (always): on the usual 80/20 split, train the logistic regression of
run.py, then boost it with small trees and print the validation F1 after
0, 10, 25, ... rounds (round 0 = logistic regression alone).

Step 2 (RUN_CV = True): 5-fold cross-validation of the chosen setting with
validation.py, on the same folds as cv_final.py -> compare with the logistic
regression alone.

Step 3 (MAKE_SUBMISSION = True): retrain on all the data, write a CSV.

Usage: python experiments_boosting.py
"""
import time

import numpy as np

from helpers import create_csv_submission
from preprocessing import split_train_val, best_threshold, apply_preprocessing
from run import load_data, build_features, train
from trees import fit_bins, apply_bins
from boosting import sigmoid, train_boosting, predict_boosting

RUN_CV = False
MAKE_SUBMISSION = False

# settings tried in step 1
CONFIGS = [
    dict(max_depth=3, learning_rate=0.1),
    dict(max_depth=5, learning_rate=0.1),
]
N_ROUNDS = 150
CHECKPOINTS = [0, 10, 25, 50, 75, 100, 150]
COMMON = dict(min_leaf=100, reg_lambda=1.0, subsample=0.5, colsample=0.5, seed=0)

# setting used in steps 2 and 3: update it after looking at step 1
FINAL = dict(max_depth=3, learning_rate=0.1, n_rounds=100, **COMMON)


def logistic_logit(x_fit, y_fit, x_list):
    """Train run.py's logistic regression on x_fit; return its logit on each x in x_list."""
    w, mean, std = train(x_fit, y_fit.astype(float))
    return [apply_preprocessing(x, mean, std) @ w for x in x_list]


def fit_predict_boosted(x_tr, y_tr, x_ev, params=FINAL):
    """Logistic regression + boosting trained on (x_tr, y_tr); probabilities on x_ev."""
    base_tr, base_ev = logistic_logit(x_tr, y_tr, [x_tr, x_ev])
    mean, edges = fit_bins(x_tr)
    p = dict(params)
    lr = p["learning_rate"]
    trees = train_boosting(apply_bins(x_tr, mean, edges), y_tr, base_tr, **p)
    return predict_boosting(trees, apply_bins(x_ev, mean, edges), base_ev, lr)


def main():
    x_train, x_test, y01, test_ids, names = load_data()

    # ---------------- Step 1: single 80/20 split ----------------
    x_tr_all, x_te_all = build_features(x_train, x_test, names)
    i_tr, i_val = split_train_val(len(x_tr_all))
    x_tr, x_val = x_tr_all[i_tr], x_tr_all[i_val]
    y_tr, y_val = y01[i_tr], y01[i_val].astype(int)

    base_tr, base_val = logistic_logit(x_tr, y_tr, [x_tr, x_val])
    mean, edges = fit_bins(x_tr)
    xb_tr, xb_val = apply_bins(x_tr, mean, edges), apply_bins(x_val, mean, edges)

    t, f = best_threshold(y_val, sigmoid(base_val))
    print(f"Logistic regression alone: F1 = {f:.4f} (threshold {t:.2f})\n")

    for cfg in CONFIGS:
        print(f"Boosting {cfg}:")
        print(f"  {'rounds':>6} {'F1':>7} {'threshold':>10}")
        t0 = time.time()

        def show(r, f_val):
            if r in CHECKPOINTS:
                t, f = best_threshold(y_val, sigmoid(f_val))
                print(f"  {r:>6} {f:>7.4f} {t:>10.2f}   ({time.time() - t0:.0f}s)", flush=True)

        train_boosting(xb_tr, y_tr, base_tr, n_rounds=N_ROUNDS, xb_val=xb_val,
                       base_val=base_val, callback=show, **cfg, **COMMON)
        print()

    # ---------------- Step 2: cross-validation ----------------
    if RUN_CV:
        from validation import cross_validate, print_cv_summary
        print(f"5-fold cross-validation of {FINAL} (several minutes)...")
        result = cross_validate(
            x_train, y01,
            lambda a, b, c, seed: fit_predict_boosted(a, b, c),
            n_splits=5, seed=0,
            preprocess=lambda a, b: build_features(a, b, names),
            threshold="inner", verbose=True)
        print_cv_summary(result)

    # ---------------- Step 3: submission ----------------
    if MAKE_SUBMISSION:
        # threshold chosen on the 80/20 split, then retrain on everything
        p_val = fit_predict_boosted(x_tr, y01[i_tr], x_val)
        t, f = best_threshold(y_val, p_val)
        print(f"Validation F1 of FINAL = {f:.4f} with threshold {t:.2f}")
        p_test = fit_predict_boosted(x_tr_all, y01, x_te_all)
        y_pred = np.where(p_test >= t, 1, -1)
        create_csv_submission(test_ids, y_pred, "submission_boosting.csv")
        print(f"submission_boosting.csv written, {np.mean(y_pred == 1):.1%} predicted positive")


if __name__ == "__main__":
    main()

Logistic regression alone: F1 = 0.4279 (threshold 0.22)

Boosting {'max_depth': 3, 'learning_rate': 0.1}:
  rounds      F1  threshold
      10  0.4300       0.21   (17s)
      25  0.4300       0.22   (36s)
      50  0.4301       0.20   (64s)
      75  0.4312       0.19   (93s)
     100  0.4322       0.21   (120s)
     150  0.4331       0.21   (177s)

Boosting {'max_depth': 5, 'learning_rate': 0.1}:
  rounds      F1  threshold
      10  0.4292       0.19   (22s)
      25  0.4309       0.22   (52s)
      50  0.4338       0.21   (98s)
      75  0.4346       0.19   (144s)
     100  0.4360       0.20   (188s)
     150  0.4362       0.20   (277s)

